# Table of Contents
## 1. Gets 1 min bars data and assigns it to 'GoodOldGoodOld.csv'
## 2. Gets sub-minute bars data and assigns it to NewGoodOldGoodOld.csv'

## 1.
## 'lastTradeDateOrContractMonth', 'durationStr', and 'endDateTime' values have to be adjusted before it is ran based on date and time period of bars you are retrieving

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import pandas as pd
from ib_insync import IB, Future
import os
from pytz import timezone
from datetime import date, timedelta, datetime
import time

# Initialize IBKR connection
ib = IB()

# Safely disconnect any existing connection
if ib.isConnected():
    print("🔌 Disconnecting previous connection...")
    ib.disconnect()

# Connect to IBKR
try:
    ib.connect('127.0.0.1', 7496, clientId=1)
    print("✅ Connected to IBKR.")
except Exception as e:
    print(f"❌ Failed to connect to IBKR: {e}")
    raise

# Define the contract
contract = Future(symbol='ES', lastTradeDateOrContractMonth='202506', exchange='CME', includeExpired= True)
ib.qualifyContracts(contract)

# Specify the file to store the dataset
existing_file = 'GoodOldGoodOld.csv'
#existing_file = 'Look.csv'

# Define the duration for fetching recent data (e.g., last 2 months)
durationStr = '1 M'

# Request historical data
try:
    bars = ib.reqHistoricalData(
        contract,
        endDateTime= '',  # Leave blank for the most recent data
        durationStr=durationStr,
        barSizeSetting='1 min',  # Adjust as needed
        whatToShow='TRADES',  # Use TRADES, MIDPOINT, etc., as needed
        useRTH= False  # Use regular trading hours
        
    )
    print("✅ Historical data fetched successfully.")
except Exception as e:
    print(f"❌ Failed to fetch historical data: {e}")
    ib.disconnect()
    raise

# Convert to DataFrame
new_data = pd.DataFrame([b.dict() for b in bars])

# 🔍 Debugging Step: Print available columns
print("\n🔍 Available columns in new_data:")
print(new_data.columns)

# Check if 'date' exists
if 'date' in new_data.columns:
    # Convert 'date' to datetime
    new_data['date'] = pd.to_datetime(new_data['date'])

    # 🔹 **Convert only if timezone is already set**
    if new_data['date'].dt.tz is not None:
        new_data['date'] = new_data['date'].dt.tz_convert('US/Eastern')
    else:
        new_data['date'] = new_data['date'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')

    # 🔹 **Filter Out Weekends (Saturday=5, Sunday=6)**
    new_data = new_data[new_data['date'].dt.weekday < 5]  # Only Monday–Friday

    # Extract time in HH:MM:SS format
    new_data['time'] = new_data['date'].dt.strftime('%H:%M:%S')

else:
    print("❌ Error: 'date' column not found. Here is a sample row from new_data:")
    print(new_data.head())
    ib.disconnect()
    raise SystemExit

# Define the specific times to keep
specific_times = ["09:30:00","09:30:00", "09:31:00", "09:32:00", "09:33:00", 
    "09:34:00", "09:35:00", "09:36:00", "09:38:00", "09:40:00", "09:43:00", 
    "09:45:00", "09:46:00", "09:50:00", "09:51:00", "09:55:00", "09:57:00", 
    "10:00:00", "10:04:00", "10:13:00", "10:15:00", "10:25:00", "10:30:00", 
    "10:40:00", "10:45:00", "10:59:00", "11:00:00", "11:15:00", "11:23:00", 
    "11:30:00", "11:54:00", "12:00:00", "12:15:00", "12:30:00", "12:32:00", 
    "12:45:00", "13:00:00", "13:15:00", "13:30:00", "13:45:00", "13:53:00", 
    "14:00:00", "14:15:00", "14:25:00", "14:30:00", "14:45:00", "15:00:00", 
    "15:15:00", "15:30:00", "15:45:00", "15:47:00", "16:00:00", "16:15:00"
]

# Filter the DataFrame to keep only the rows with the specified times
filtered_data = new_data[new_data['time'].isin(specific_times)]

print(filtered_data)

# Check if the file already exists
if os.path.exists(existing_file):
    # Load the existing dataset
    existing_data = pd.read_csv(existing_file)
    existing_data['date'] = pd.to_datetime(existing_data['date'])

    # Combine new data with the existing dataset, keeping only unique rows
    combined_data = pd.concat([existing_data, filtered_data]).drop_duplicates(subset='date', keep='last')
else:
    # If the file doesn't exist, use the filtered data as the dataset
    combined_data = filtered_data

# Sort by date before saving
combined_data.sort_values(by=['date'], inplace=True)

# Save the updated dataset to the file
combined_data.to_csv(existing_file, index=False)
print(f"✅ Dataset updated and saved to '{existing_file}'.")

# Disconnect from IBKR
if ib.isConnected():
    print("🔌 Disconnecting from IBKR...")
    ib.disconnect()
    print("✅ Disconnected")
else:
    print("⚠️ No existing connection found.")

## 2.
## 'lastTradeDateOrContractMonth', 'start', and 'end' values have to be adjusted before it is ran based on date and time period of bars you are retrieving.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import pandas as pd
from ib_insync import IB, Future
import os
from pytz import timezone
from datetime import date, timedelta, datetime
import time

# Initialize IBKR connection
ib = IB()

# Safely disconnect any existing connection
if ib.isConnected():
    print("🔌 Disconnecting previous connection...")
    ib.disconnect()

# Connect to IBKR
try:
    ib.connect('127.0.0.1', 7496, clientId=1)
    print("✅ Connected to IBKR.")
except Exception as e:
    print(f"❌ Failed to connect to IBKR: {e}")
    raise

# Define the contract
contract = Future(symbol='ES', lastTradeDateOrContractMonth='202506', exchange='CME', includeExpired= True)
ib.qualifyContracts(contract)

# Specify the file to store the dataset
existing_file = 'NewGoodOldGoodOld.csv'
#existing_file = 'Look.csv'

# Define the duration for fetching recent data (e.g., last 2 months)

from datetime import datetime, timedelta

def iterate_weekdays(start_date, end_date):
    current_date = start_date
    while current_date <= end_date:
        if current_date.weekday() < 5:  # Monday–Friday only (0=Monday, 4=Friday)
            yield current_date
        current_date += timedelta(days=1)

start = datetime(2025, 3, 6)
end = datetime(2025, 3, 21)

for date in iterate_weekdays(start, end):
    endDate= date.strftime('%Y%m%d')
    print(endDate)
    endDateTimeStr= endDate +  ' 10:00:00 America/New_York'
    print(endDateTimeStr)

# Request historical data
    try:
        
        bars = ib.reqHistoricalData(
            contract,
            endDateTime=endDateTimeStr,
            durationStr="1800 S",    # 09:30 → 10:00 => 30 minutes
            barSizeSetting="1 secs",
            whatToShow='TRADES',
            useRTH=True,
        )
    
        ib.sleep(10)  # ✅ Sleep to prevent pacing violations
        
        print("✅ Historical data fetched successfully.")
    except Exception as e:
        print(f"❌ Failed to fetch historical data: {e}")
        ib.disconnect()
        raise

    # Convert to DataFrame
    new_data = pd.DataFrame([b.dict() for b in bars])
    
    # 🔍 Debugging Step: Print available columns
    print("\n🔍 Available columns in new_data:")
    print(new_data.columns)
    
    # Check if 'date' exists
    if 'date' in new_data.columns:
        # Convert 'date' to datetime
        new_data['date'] = pd.to_datetime(new_data['date'])
    
        # 🔹 **Convert only if timezone is already set**
        if new_data['date'].dt.tz is not None:
            new_data['date'] = new_data['date'].dt.tz_convert('US/Eastern')
        else:
            new_data['date'] = new_data['date'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
    
        # 🔹 **Filter Out Weekends (Saturday=5, Sunday=6)**
        new_data = new_data[new_data['date'].dt.weekday < 5]  # Only Monday–Friday
    
        # Extract time in HH:MM:SS format
        new_data['time'] = new_data['date'].dt.strftime('%H:%M:%S')
    
    else:
        print("❌ Error: 'date' column not found. Here is a sample row from new_data:")
        print(new_data.head())
        ib.disconnect()
        raise SystemExit
    
    # Define the specific times to keep
    specific_times = [
        "09:30:01", "09:30:02", "09:30:03", "09:30:05", "09:30:08", "09:30:10",
        "09:30:11", "09:30:13", "09:30:15", "09:30:20", "09:30:21", "09:30:25", 
        "09:30:30", "09:30:34", "09:30:35", "09:30:40", "09:30:45", "09:30:50",
        "09:30:55"
    ]
    
    # Filter the DataFrame to keep only the rows with the specified times
    filtered_data = new_data[new_data['time'].isin(specific_times)]
    
    # Check if the file already exists
    if os.path.exists(existing_file):
        # Load the existing dataset
        existing_data = pd.read_csv(existing_file)
        existing_data['date'] = pd.to_datetime(existing_data['date'])
    
        # Combine new data with the existing dataset, keeping only unique rows
        combined_data = pd.concat([existing_data, filtered_data]).drop_duplicates(subset='date', keep='last')
    else:
        # If the file doesn't exist, use the filtered data as the dataset
        combined_data = filtered_data
    
    # Sort by date before saving
    combined_data.sort_values(by=['date'], inplace=True)
    
    # Save the updated dataset to the file
    combined_data.to_csv(existing_file, index=False)
    print(f"✅ Dataset updated and saved to '{existing_file}'.")

# Disconnect from IBKR
if ib.isConnected():
    print("🔌 Disconnecting from IBKR...")
    ib.disconnect()
    print("✅ Disconnected")
else:
    print("⚠️ No existing connection found.")